# EX_05 — Vector stores y retrieval (ejercicios)

**Notebook de referencia:** `notebook/05_Vectorstores_Retrieval.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Chunking

Implementa un chunker trivial por **número de caracteres** con solapamiento (`chunk_size`, `chunk_overlap`). Aplícalo a un texto largo en una lista de strings.


In [1]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    chunks = []
    
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        
        start += chunk_size - overlap
    
    return chunks


long_text = "word " * 500

chunks = chunk_text(long_text, chunk_size=200, overlap=40)

print("Number of chunks:", len(chunks))
print("First chunk:")
print(chunks[0])
print("\nSecond chunk:")
print(chunks[1])

Number of chunks: 16
First chunk:
word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word 

Second chunk:
word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word 


## Actividad 2 — Embeddings + FAISS

Embedde los chunks (puede ser `sentence_transformers`) y construye un índice `faiss.IndexFlatIP` o `IndexFlatL2`. Recupera los top-3 para una query.

*Hint:* L2-normalize vectors if you treat inner product as cosine similarity.


In [2]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed chunks
embeddings = model.encode(chunks)

# Convert to float32 because FAISS requires it
embeddings = np.array(embeddings).astype("float32")

# Normalize vectors to use inner product as cosine similarity
faiss.normalize_L2(embeddings)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Add embeddings to index
index.add(embeddings)

# Query
query = "word word word"
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")
faiss.normalize_L2(query_embedding)

# Search top-3
scores, indices = index.search(query_embedding, k=3)

print("Top-3 results:")
for score, idx in zip(scores[0], indices[0]):
    print(f"\nIndex: {idx}")
    print(f"Score: {score}")
    print(chunks[idx][:200])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top-3 results:

Index: 15
Score: 0.8334558606147766
word word word word word word word word word word word word word word word word word word word word 

Index: 2
Score: 0.7508412599563599
word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word 

Index: 1
Score: 0.7508412599563599
word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word word 


## Actividad 3 — Métrica manual

Para una query y tres documentos **artificiales** (uno relevante, dos ruido), muestra scores de similitud y verifica que el relevante queda primero.


In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

query = "How can I use vector databases for semantic search?"

documents = [
    "Vector databases store embeddings and allow semantic search over documents.",
    "Pizza dough needs flour, water, yeast and salt.",
    "The weather today is sunny with mild temperatures."
]

# Encode query and documents
query_embedding = model.encode(query)
doc_embeddings = model.encode(documents)

# Cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = []

for i, doc_embedding in enumerate(doc_embeddings):
    score = cosine_similarity(query_embedding, doc_embedding)
    scores.append((i, score, documents[i]))

# Sort by similarity score
scores = sorted(scores, key=lambda x: x[1], reverse=True)

print("Ranking results:")

for rank, (idx, score, doc) in enumerate(scores, start=1):
    print(f"\nRank {rank}")
    print(f"Document index: {idx}")
    print(f"Score: {score:.4f}")
    print(f"Document: {doc}")

print("\nMost relevant document:")
print(scores[0][2])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ranking results:

Rank 1
Document index: 0
Score: 0.7979
Document: Vector databases store embeddings and allow semantic search over documents.

Rank 2
Document index: 2
Score: 0.0011
Document: The weather today is sunny with mild temperatures.

Rank 3
Document index: 1
Score: -0.0330
Document: Pizza dough needs flour, water, yeast and salt.

Most relevant document:
Vector databases store embeddings and allow semantic search over documents.
